In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_train_clean3.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '3_val_clean3.parquet')

print(train.shape, val.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(616293, 35) (133316, 35)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 1. 데이터 불러오기 및 Tidal / Normal 분리
# ============================================================

# 1-2. 조석 주기성 후보 관측소 9개
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# 1-3. Train 분리
tidal_train = train.loc[
    train['station_id'].isin(tidal_stations)
].copy()

normal_train = train.loc[
    ~train['station_id'].isin(tidal_stations)
].copy()

# 1-4. Validation 분리
tidal_val = val.loc[
    val['station_id'].isin(tidal_stations)
].copy()

normal_val = val.loc[
    ~val['station_id'].isin(tidal_stations)
].copy()

# 1-5. 데이터 분리 검증
assert len(tidal_train) + len(normal_train) == len(train)
assert len(tidal_val) + len(normal_val) == len(val)

print(f'Train: {len(train):,}')
print(f'  Tidal: {len(tidal_train):,}')
print(f'  Normal: {len(normal_train):,}')

print(f'\nValidation: {len(val):,}')
print(f'  Tidal: {len(tidal_val):,}')
print(f'  Normal: {len(normal_val):,}')

Train: 616,293
  Tidal: 11,685
  Normal: 604,608

Validation: 133,316
  Tidal: 2,566
  Normal: 130,750


In [5]:
# ============================================================
# 2. Tidal / Normal 독립변수 및 종속변수 분리
# ============================================================

# 2-1. Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
y_tidal_train = tidal_train['target_maxrise_6h']

# 2-2. Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
y_normal_train = normal_train['target_maxrise_6h']

# 2-3. CatBoost 범주형 변수 처리
# K-Fold할거라 val셋은 생성 안함
X_tidal_train['station_id'] = X_tidal_train['station_id'].astype(str)
X_normal_train['station_id'] = X_normal_train['station_id'].astype(str)

# 2-4. 독립 Validation 데이터
X_tidal_val = tidal_val[num_cols_fe + ['station_id']].copy()
y_tidal_val = tidal_val['target_maxrise_6h']

X_normal_val = normal_val[num_cols_fe + ['station_id']].copy()
y_normal_val = normal_val['target_maxrise_6h']

X_tidal_val['station_id'] = X_tidal_val['station_id'].astype(str)
X_normal_val['station_id'] = X_normal_val['station_id'].astype(str)

In [7]:
# ============================================================
# 3. 시간사상 Group K-Fold 설정: 기존 Fold 배정 재사용
# ============================================================

# 3-1. Group 설정
kf_groups = train['event_id']

print(f'전체 학습 행 수 : {len(train):,}')
print(f'시간사상 개수   : {kf_groups.nunique():,}')

assert kf_groups.notna().all(), 'event_id 결측치 존재'
assert train.index.is_unique, 'Train 인덱스 중복'


# 3-2. 기존 Fold 배정 CSV 불러오기
N_SPLITS = 5

kf_fold_f = pd.read_csv(
    PROJECT_ROOT / 'config' / '0_kf_event_fold_match.csv'
)

# event_id 중복 여부 확인 (시간사상별 Fold 배정은 1개여야 함)
assert kf_fold_f['event_id'].is_unique

# CSV 전체 데이터에 결측치가 없는지 확인
assert not kf_fold_f.isna().any().any()

# 모든 Fold 번호가 1~5 범위에 있는지 확인
assert kf_fold_f['fold_num'].between(1, N_SPLITS).all()

# CSV와 Train의 시간사상 목록이 정확히 일치하는지 확인
# 집합(set) 비교이므로 순서와 중복 횟수는 무시
assert set(kf_fold_f['event_id']) == set(kf_groups)


# 3-3. 기존 Fold 번호 대응
# 'event_id'로 매칭해서 row_id 인덱스에 fold_num시리즈 생성
kf_fold_num = kf_groups.map(
    kf_fold_f.set_index('event_id')['fold_num']
)

kf_fold_num.name = 'fold_num'  # 이거없으면 제목이 'event_id'로 유지됨

assert kf_fold_num.notna().all(), 'Fold 배정 누락'


# 3-4. Tidal / Normal Fold 번호 대응
# reindex로 인덱스 있는것만 뽑아서 row_id 인덱스에 fold_num시리즈 필터링
tidal_fold_num = kf_fold_num.reindex(X_tidal_train.index)
normal_fold_num = kf_fold_num.reindex(X_normal_train.index)

assert tidal_fold_num.notna().all()
assert normal_fold_num.notna().all()

print('\n<Tidal Fold별 데이터 개수>')
print(tidal_fold_num.value_counts().sort_index())

print('\n<Normal Fold별 데이터 개수>')
print(normal_fold_num.value_counts().sort_index())

전체 학습 행 수 : 616,293
시간사상 개수   : 98

<Tidal Fold별 데이터 개수>
fold_num
1    2377
2    2297
3    2347
4    2338
5    2326
Name: count, dtype: int64

<Normal Fold별 데이터 개수>
fold_num
1    120619
2    121102
3    121052
4    120878
5    120957
Name: count, dtype: int64


In [8]:
# ============================================================
# 4. Tidal CatBoost 5-Fold 학습 및 OOF 생성
# ============================================================

# 4-1. 결과 저장용 변수
tidal_kf_results = []

tidal_kf_oof_pred = pd.Series(  # train셋 분할 fold별 예측값 담을 것
    index=X_tidal_train.index,
    dtype=float
)

tidal_kf_val_preds = pd.DataFrame(
    index=X_tidal_val.index
)

# 4-2. Fold별 학습
for fold_num in range(1, N_SPLITS + 1):

    print(f'\n========== Tidal Fold {fold_num} ==========')

    # 기존 Fold 번호로 학습·검증 분할
    train_mask = tidal_fold_num != fold_num
    valid_mask = tidal_fold_num == fold_num

    X_kf_tr = X_tidal_train.loc[train_mask]
    X_kf_va = X_tidal_train.loc[valid_mask]

    y_kf_tr = y_tidal_train.loc[train_mask]
    y_kf_va = y_tidal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_va) > 0

    # 모델 정의
    model_cat_kf = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']
    )

    # 학습
    model_cat_kf.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 4-3. OOF 예측
    pred_kf_va = model_cat_kf.predict(X_kf_va)
    pred_kf_tr = model_cat_kf.predict(X_kf_tr)

    tidal_kf_oof_pred.loc[X_kf_va.index] = pred_kf_va

    # 4-4. 독립 Validation 전체 예측 및 Fold별 저장
    tidal_kf_val_preds[f'fold_{fold_num}'] = model_cat_kf.predict(
        X_tidal_val
    )

    # 4-5. Fold별 평가
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    tidal_kf_results.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_cat_kf.best_iteration_
    })

    print(
        f'Tidal Fold {fold_num} | '
        f'OOF={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_cat_kf.best_iteration_}'
    )

# 4-6. 전체 OOF 평가
assert tidal_kf_oof_pred.notna().all(), 'Tidal OOF 예측 누락'
assert tidal_kf_val_preds.shape[1] == N_SPLITS
assert tidal_kf_val_preds.notna().all().all(), 'Validation 예측 누락'

tidal_kf_oof_rmse = root_mean_squared_error(
    y_tidal_train,
    tidal_kf_oof_pred
)

tidal_kf_results = pd.DataFrame(tidal_kf_results)

print('\n<Tidal Fold별 학습 결과>')
display(tidal_kf_results)

print(f'\nTidal 전체 OOF RMSE : {tidal_kf_oof_rmse:.6f}')


========== Tidal Fold 1 ==========
Learning rate set to 0.03698
0:	learn: 1.6741950	test: 1.8746893	best: 1.8746893 (0)	total: 178ms	remaining: 8m 54s
200:	learn: 0.3966562	test: 0.5275649	best: 0.5275649 (200)	total: 7.74s	remaining: 1m 47s
400:	learn: 0.3374297	test: 0.4975205	best: 0.4975205 (400)	total: 16.2s	remaining: 1m 45s
600:	learn: 0.3016497	test: 0.4840770	best: 0.4840770 (600)	total: 24.8s	remaining: 1m 39s
800:	learn: 0.2775717	test: 0.4791114	best: 0.4788893 (770)	total: 32.5s	remaining: 1m 29s
1000:	learn: 0.2575283	test: 0.4771059	best: 0.4768975 (947)	total: 40.6s	remaining: 1m 21s
1200:	learn: 0.2401385	test: 0.4761404	best: 0.4751086 (1169)	total: 47.8s	remaining: 1m 11s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.4751086213
bestIteration = 1169

Shrink model to first 1170 iterations.
Tidal Fold 1 | OOF=0.4751 | Train=0.2427 | Gap=0.2324 | Best iter=1169

========== Tidal Fold 2 ==========
Learning rate set to 0.037029
0:	learn: 1.7144391	t

,fold,val_rmse,train_rmse,gap,best_iteration
0,1,0.475109,0.242733,0.232376,1169
1,2,0.359429,0.211879,0.147550,1651
2,3,0.344497,0.154577,0.189920,2975
3,4,0.372886,0.157071,0.215815,2918
4,5,0.363554,0.159285,0.204270,2819



Tidal 전체 OOF RMSE : 0.386370


In [9]:
# ============================================================
# 5. Normal CatBoost 5-Fold 학습 및 OOF 생성
# ============================================================

# 5-1. 결과 저장용 변수
normal_kf_results = []

normal_kf_oof_pred = pd.Series(
    index=X_normal_train.index,
    dtype=float
)

normal_kf_val_preds = pd.DataFrame(
    index=X_normal_val.index
)

# 5-2. Fold별 학습
for fold_num in range(1, N_SPLITS + 1):

    print(f'\n========== Normal Fold {fold_num} ==========')

    # 기존 Fold 번호로 학습·검증 분할
    train_mask = normal_fold_num != fold_num
    valid_mask = normal_fold_num == fold_num

    X_kf_tr = X_normal_train.loc[train_mask]
    X_kf_va = X_normal_train.loc[valid_mask]

    y_kf_tr = y_normal_train.loc[train_mask]
    y_kf_va = y_normal_train.loc[valid_mask]

    assert len(X_kf_tr) > 0 and len(X_kf_va) > 0

    # 모델 정의
    model_cat_kf = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']
    )

    # 학습
    model_cat_kf.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 5-3. OOF 예측
    pred_kf_va = model_cat_kf.predict(X_kf_va)
    pred_kf_tr = model_cat_kf.predict(X_kf_tr)

    normal_kf_oof_pred.loc[X_kf_va.index] = pred_kf_va

    # 5-4. 독립 Validation 전체 예측 및 Fold별 저장
    normal_kf_val_preds[f'fold_{fold_num}'] = model_cat_kf.predict(
        X_normal_val
    )

    # 5-5. Fold별 평가
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    normal_kf_results.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_cat_kf.best_iteration_
    })

    print(
        f'Normal Fold {fold_num} | '
        f'OOF={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_cat_kf.best_iteration_}'
    )

# 5-6. 전체 OOF 평가
assert normal_kf_oof_pred.notna().all(), 'Normal OOF 예측 누락'
assert normal_kf_val_preds.shape[1] == N_SPLITS
assert normal_kf_val_preds.notna().all().all(), 'Validation 예측 누락'

normal_kf_oof_rmse = root_mean_squared_error(
    y_normal_train,
    normal_kf_oof_pred
)

normal_kf_results = pd.DataFrame(normal_kf_results)

print('\n<Normal Fold별 학습 결과>')
display(normal_kf_results)

print(f'\nNormal 전체 OOF RMSE : {normal_kf_oof_rmse:.6f}')


========== Normal Fold 1 ==========
Learning rate set to 0.068766
0:	learn: 0.2752341	test: 0.3199435	best: 0.3199435 (0)	total: 187ms	remaining: 9m 20s
200:	learn: 0.2055879	test: 0.2694372	best: 0.2694372 (200)	total: 37.7s	remaining: 8m 45s
400:	learn: 0.1950586	test: 0.2660327	best: 0.2660327 (400)	total: 1m 10s	remaining: 7m 36s
600:	learn: 0.1885954	test: 0.2653412	best: 0.2653038 (595)	total: 1m 48s	remaining: 7m 12s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2649676576
bestIteration = 690

Shrink model to first 691 iterations.
Normal Fold 1 | OOF=0.2650 | Train=0.1852 | Gap=0.0797 | Best iter=690

========== Normal Fold 2 ==========
Learning rate set to 0.068755
0:	learn: 0.3077417	test: 0.1648772	best: 0.1648772 (0)	total: 166ms	remaining: 8m 16s
200:	learn: 0.2334563	test: 0.1418200	best: 0.1418200 (200)	total: 33.8s	remaining: 7m 50s
400:	learn: 0.2218892	test: 0.1394775	best: 0.1394703 (396)	total: 1m 9s	remaining: 7m 31s
600:	learn: 0.2147261	tes

,fold,val_rmse,train_rmse,gap,best_iteration
0,1,0.264968,0.185222,0.079746,690
1,2,0.138159,0.206136,-0.067977,896
2,3,0.222469,0.174849,0.047620,2153
3,4,0.268136,0.187453,0.080682,566
4,5,0.214383,0.189199,0.025184,1118



Normal 전체 OOF RMSE : 0.226512


In [10]:
# ============================================================
# 9. Tidal + Normal 통합 OOF 및 Validation RMSE 평가
# ============================================================

from sklearn.metrics import root_mean_squared_error

# 9-1. 전체 Train OOF 예측값 통합
full_oof_pred = pd.concat([
    tidal_kf_oof_pred,
    normal_kf_oof_pred
])

# 9-2. Tidal / Normal 5-Fold Validation 평균 앙상블
tidal_kf_val_ensemble = tidal_kf_val_preds.mean(axis=1)
normal_kf_val_ensemble = normal_kf_val_preds.mean(axis=1)

# 전체 Validation 앙상블 예측값 통합
full_val_pred = pd.concat([
    tidal_kf_val_ensemble,
    normal_kf_val_ensemble
])

# 9-3. 통합 예측값 검증
assert full_oof_pred.index.is_unique, 'OOF 인덱스 중복'
assert full_val_pred.index.is_unique, 'Validation 인덱스 중복'

assert set(full_oof_pred.index) == set(train.index), 'Train 인덱스 불일치'
assert set(full_val_pred.index) == set(val.index), 'Validation 인덱스 불일치'

assert not full_oof_pred.isnull().any(), '전체 OOF 예측 누락'
assert not full_val_pred.isnull().any(), '전체 Validation 예측 누락'

# 9-4. 원본 인덱스 순서로 예측값 정렬
full_oof_pred = full_oof_pred.reindex(train.index)
full_val_pred = full_val_pred.reindex(val.index)

print('전체 OOF 예측 행 수       :', len(full_oof_pred))
print('전체 Validation 예측 행 수:', len(full_val_pred))

# 9-5. 전체 RMSE 계산
full_oof_rmse = root_mean_squared_error(
    train['target_maxrise_6h'],
    full_oof_pred
)

full_val_rmse = root_mean_squared_error(
    val['target_maxrise_6h'],
    full_val_pred
)

# Tidal / Normal 개별 Validation RMSE
tidal_kf_val_rmse = root_mean_squared_error(
    y_tidal_val,
    tidal_kf_val_ensemble
)

normal_kf_val_rmse = root_mean_squared_error(
    y_normal_val,
    normal_kf_val_ensemble
)

# 9-6. 최종 성능 비교표
final_results = pd.DataFrame({
    'OOF_RMSE': [
        tidal_kf_oof_rmse,
        normal_kf_oof_rmse,
        full_oof_rmse
    ],
    'Val_RMSE': [
        tidal_kf_val_rmse,
        normal_kf_val_rmse,
        full_val_rmse
    ]
}, index=['Tidal', 'Normal', '전체'])

print('\n<최종 모델 성능 비교>')
display(final_results)

전체 OOF 예측 행 수       : 616293
전체 Validation 예측 행 수: 133316

<최종 모델 성능 비교>


,OOF_RMSE,Val_RMSE
Tidal,0.386370,0.339517
Normal,0.226512,0.185111
전체,0.230576,0.189276


In [13]:
# ============================================================
# 6. Tidal / Normal K-Fold 예측 결과 저장
# ============================================================

# Tidal OOF 및 Validation 예측 저장
tidal_kf_oof_pred.to_frame('oof_pred').to_parquet(  # to_frame: 시리즈를 df로 저장(oof는 단일 컬럼이라 시리즈임)
    PROJECT_ROOT / 'parqueted_data' / 'kf_results' / 'tidal_kf_oof_pred.parquet'
)

tidal_kf_val_preds.to_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'kf_results' / 'tidal_kf_val_preds.parquet'
)

# Normal OOF 및 Validation 예측 저장
normal_kf_oof_pred.to_frame('oof_pred').to_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'kf_results' / 'normal_kf_oof_pred.parquet'
)

normal_kf_val_preds.to_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'kf_results' / 'normal_kf_val_preds.parquet'
)

print('Tidal / Normal K-Fold 예측 결과 저장 완료')

Tidal / Normal K-Fold 예측 결과 저장 완료


In [ ]:
# ============================================================
# 6-1. 저장된 K-Fold 예측 결과 불러오기
# ============================================================

tidal_kf_oof_pred = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'tidal_kf_oof_pred.parquet'
)['oof_pred']

tidal_kf_val_preds = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'tidal_kf_val_preds.parquet'
)

normal_kf_oof_pred = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'normal_kf_oof_pred.parquet'
)['oof_pred']

normal_kf_val_preds = pd.read_parquet(
    PROJECT_ROOT / 'parqueted_data' / 'normal_kf_val_preds.parquet'
)

print('K-Fold 예측 결과 불러오기 완료')

In [ ]:
# ============================================================
# 6. Tidal / Normal OOF 예측값 결합 및 전체 RMSE
# ============================================================

# 인덱스 중복 검증
assert train.index.is_unique, 'Train 인덱스 중복'

# 전체 Train의 row_id를 기준으로 OOF 저장
kf_oof_pred = pd.Series(index=train.index, dtype=float)

# Tidal / Normal OOF 예측값 결합
kf_oof_pred.loc[tidal_kf_oof_pred.index] = tidal_kf_oof_pred
kf_oof_pred.loc[normal_kf_oof_pred.index] = normal_kf_oof_pred

# 예측값 누락 검증
assert kf_oof_pred.notna().all(), 'OOF 예측값 누락'

# 전체 OOF RMSE
kf_oof_rmse = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred
)

print(f'전체 Train OOF RMSE : {kf_oof_rmse:.6f}')

In [ ]:
# ============================================================
# 5. Tidal + Normal OOF 예측값 결합 및 전체 RMSE
# ============================================================

# 1. OOF 예측값 결합
kf_oof_pred = pd.Series(index=train.index, dtype=float)

kf_oof_pred.loc[tidal_train.index] = tidal_kf_oof_pred
kf_oof_pred.loc[normal_train.index] = normal_kf_oof_pred

# 2. 예측 누락 검증
assert kf_oof_pred.notna().all()

# 3. 전체 OOF RMSE
oof_rmse = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred
)

print(f'Tidal OOF RMSE  : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Normal OOF RMSE : {root_mean_squared_error(y_normal_train, normal_kf_oof_pred):.6f}')
print(f'전체 OOF RMSE   : {oof_rmse:.6f}')

In [ ]:
# ============================================================
# 6. Tidal 모델 하이퍼파라미터 실험 (depth=4)
# ============================================================

# 1. OOF 예측값 및 결과 저장
tidal_kf_oof_pred_d4 = np.full(len(X_tidal_train), np.nan)
tidal_kf_results_d4 = []

# 2. Tidal 5-Fold 학습
for fold_num, (train_idx, val_idx) in enumerate(
    tidal_kf_splits,
    start=1
):
    # Fold별 데이터 분할
    X_kf_tr = X_tidal_train.iloc[train_idx]
    X_kf_va = X_tidal_train.iloc[val_idx]

    y_kf_tr = y_tidal_train.iloc[train_idx]
    y_kf_va = y_tidal_train.iloc[val_idx]

    # 모델 정의 (depth=4)
    model_tidal_d4 = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        depth=4,
        cat_features=['station_id']
    )

    # 학습
    model_tidal_d4.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 예측
    pred_kf_va = model_tidal_d4.predict(X_kf_va)
    pred_kf_tr = model_tidal_d4.predict(X_kf_tr)

    tidal_kf_oof_pred_d4[val_idx] = pred_kf_va

    # RMSE 계산
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    tidal_kf_results_d4.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_tidal_d4.best_iteration_
    })

    print(
        f'Fold {fold_num} | '
        f'Val={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_tidal_d4.best_iteration_}'
    )

# 3. Fold별 결과표
tidal_kf_results_d4 = pd.DataFrame(tidal_kf_results_d4)

display(tidal_kf_results_d4)

# 4. Tidal OOF RMSE
assert not np.isnan(tidal_kf_oof_pred_d4).any()

rmse_tidal_d4 = root_mean_squared_error(
    y_tidal_train,
    tidal_kf_oof_pred_d4
)

# 5. 기존 Normal OOF와 결합
kf_oof_pred_d4 = pd.Series(index=train.index, dtype=float)

kf_oof_pred_d4.loc[tidal_train.index] = tidal_kf_oof_pred_d4
kf_oof_pred_d4.loc[normal_train.index] = normal_kf_oof_pred

assert kf_oof_pred_d4.notna().all()

# 6. 전체 OOF RMSE
rmse_split_d4 = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred_d4
)

# 7. 기존 모델과 비교
print('\n========== OOF RMSE 비교 ==========')

print(f'Tidal 기존 depth=6 : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Tidal 신규 depth=4 : {rmse_tidal_d4:.6f}')
print()
print(f'전체 기존 depth=6 : {oof_rmse:.6f}')
print(f'전체 신규 depth=4 : {rmse_split_d4:.6f}')
print(f'전체 RMSE 변화    : {rmse_split_d4 - oof_rmse:+.6f}')

In [ ]:
# ============================================================
# 6. Tidal 모델 하이퍼파라미터 실험 (l2_leaf_reg=10)
# ============================================================

# 1. OOF 예측값 및 결과 저장
tidal_kf_oof_pred_l2 = np.full(len(X_tidal_train), np.nan)
tidal_kf_results_l2 = []

# 2. Tidal 5-Fold 학습
for fold_num, (train_idx, val_idx) in enumerate(
    tidal_kf_splits,
    start=1
):
    # Fold별 데이터 분할
    X_kf_tr = X_tidal_train.iloc[train_idx]
    X_kf_va = X_tidal_train.iloc[val_idx]

    y_kf_tr = y_tidal_train.iloc[train_idx]
    y_kf_va = y_tidal_train.iloc[val_idx]

    # 모델 정의 (l2_leaf_reg=10)
    model_tidal_l2 = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        depth=6,
        l2_leaf_reg=10,
        cat_features=['station_id']
    )

    # 학습
    model_tidal_l2.fit(
        X_kf_tr,
        y_kf_tr,
        early_stopping_rounds=100,
        eval_set=(X_kf_va, y_kf_va),
        verbose=200
    )

    # 예측
    pred_kf_va = model_tidal_l2.predict(X_kf_va)
    pred_kf_tr = model_tidal_l2.predict(X_kf_tr)

    tidal_kf_oof_pred_l2[val_idx] = pred_kf_va

    # RMSE 계산
    val_rmse = root_mean_squared_error(y_kf_va, pred_kf_va)
    train_rmse = root_mean_squared_error(y_kf_tr, pred_kf_tr)

    tidal_kf_results_l2.append({
        'fold': fold_num,
        'val_rmse': val_rmse,
        'train_rmse': train_rmse,
        'gap': val_rmse - train_rmse,
        'best_iteration': model_tidal_l2.best_iteration_
    })

    print(
        f'Fold {fold_num} | '
        f'Val={val_rmse:.4f} | '
        f'Train={train_rmse:.4f} | '
        f'Gap={val_rmse - train_rmse:.4f} | '
        f'Best iter={model_tidal_l2.best_iteration_}'
    )

# 3. Fold별 결과표
tidal_kf_results_l2 = pd.DataFrame(tidal_kf_results_l2)

display(tidal_kf_results_l2)

# 4. Tidal OOF RMSE
assert not np.isnan(tidal_kf_oof_pred_l2).any()

rmse_tidal_l2 = root_mean_squared_error(
    y_tidal_train,
    tidal_kf_oof_pred_l2
)

# 5. 기존 Normal OOF와 결합
kf_oof_pred_l2 = pd.Series(index=train.index, dtype=float)

kf_oof_pred_l2.loc[tidal_train.index] = tidal_kf_oof_pred_l2
kf_oof_pred_l2.loc[normal_train.index] = normal_kf_oof_pred

assert kf_oof_pred_l2.notna().all()

# 6. 전체 OOF RMSE
rmse_split_l2 = root_mean_squared_error(
    train['target_maxrise_6h'],
    kf_oof_pred_l2
)

# 7. 기존 모델과 비교
print('\n========== OOF RMSE 비교 ==========')

print(f'Tidal 기존 l2=3  : {root_mean_squared_error(y_tidal_train, tidal_kf_oof_pred):.6f}')
print(f'Tidal 신규 l2=10 : {rmse_tidal_l2:.6f}')
print()
print(f'전체 기존 l2=3  : {oof_rmse:.6f}')
print(f'전체 신규 l2=10 : {rmse_split_l2:.6f}')
print(f'전체 RMSE 변화  : {rmse_split_l2 - oof_rmse:+.6f}')